# تجربة الصوت التجريبية — حكاية سميرة موسى (010)
تشغيل مجاني عبر Google Colab باستخدام VoiceTut-TTS باللهجة المصرية. لا تُرفع أي ملفات صوتية إلى GitHub تلقائيًا.

**مهم:** Sarah صوت نسائي مصري بالغ ضمن الأصوات المدمجة، وليس صوت طفلة حقيقيًا مثبتًا. Mohammed صوت رجالي مصري للراوي. استمع للعينة أولًا، ولا تنشر الصوت قبل مراجعة شروط النموذج والأصوات وحقوق الاستخدام. النموذج نفسه مرخص Apache-2.0 وفق بطاقة النموذج، لكن ذلك لا يغني عن مراجعة حقوق عينات الأصوات المدمجة.


## 1) تفعيل GPU
من قائمة Runtime اختر Change runtime type ثم T4 GPU إن كان متاحًا. التشغيل المجاني وتوفر GPU يخضعان لسياسة Google Colab.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('تنبيه: اختر GPU من إعدادات Colab قبل تحميل النموذج؛ التشغيل على CPU قد يكون بطيئًا جدًا.')

In [ ]:
!pip -q install git+https://github.com/k2-fsa/OmniVoice.git voicetut-tts soundfile

In [ ]:
from voicetut_tts import VoiceTutTTS
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
from pathlib import Path

tts = VoiceTutTTS.from_pretrained('mohammedaly22/VoiceTut-TTS')
print('تم تحميل نموذج VoiceTut-TTS')

## 2) نص الحوار المختصر
هذا مقتطف من الحكاية الأصلية دون تعديل في نص الحكاية داخل الموقع.

In [ ]:
# Sarah صوت نسائي مصري، وMohammed صوت رجالي مصري. لا يوجد ادعاء بأن Sarah صوت طفلة حقيقية.
speakers = {'karma': 'Sarah', 'narrator': 'Mohammed'}
outputs = []
for i, (role, text) in enumerate(dialogue, start=1):
    path = f'/content/sameera_{i:02d}_{role}.wav'
    tts.synthesize(text, speaker=speakers[role], output=path)
    audio, sr = sf.read(path)
    outputs.append((role, audio, sr, path))
    print(i, role, speakers[role], 'duration:', round(len(audio)/sr, 2), 'sec', 'sample rate:', sr)


In [ ]:
# استمع لكل دور منفصلًا
for role, audio, sr, path in outputs:
    print(role, path)
    display(Audio(audio, rate=sr))

In [ ]:
# دمج الحوار بالتبادل مع فواصل قصيرة، مع التحقق من توحيد معدل العينات.
sample_rates = {item[2] for item in outputs}
if len(sample_rates) != 1:
    raise ValueError(f'اختلاف sample rates بين المقاطع: {sample_rates}')
sr = sample_rates.pop()
parts = []
for role, audio, item_sr, path in outputs:
    parts.extend([audio.astype(np.float32), np.zeros(int(sr * 0.35), dtype=np.float32)])
mixed = np.concatenate(parts)
final_path = '/content/sameera_010_voice_pilot.wav'
sf.write(final_path, mixed, sr)
print('الملف التجريبي جاهز:', final_path, 'sample rate:', sr)
display(Audio(mixed, rate=sr))


In [ ]:
from google.colab import files
files.download('/content/sameera_010_voice_pilot.wav')